# G6M2: Baseline vs Huber vs Ridge vs Huber+Ridge (Unified Comparator)

This notebook runs 4 linear-term models on the same preprocessed data and compares them with `UnifiedModelComparator`.

- Baseline: `Urc1` (OLS)
- Huber: `Urc1BaseHuber(epsilon=1.00, alpha=0.0)`
- Ridge: `Urc1BaseHuber(epsilon=1e6, alpha=0.1)`
- Huber+Ridge: `Urc1BaseHuber(epsilon=1.00, alpha=0.1)`

Comparison enables:
- Cond metrics: `include_all_cond_metrics=True`
- GT metrics: `include_gt_metrics=True` (if GT bundle is found)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

from degradation_toolbox.Urc.Urc1 import Urc1
from degradation_toolbox.Urc.Urc1_huber import Urc1BaseHuber
from master_arbeit_Di.explore.UnifiedModelComparator import UnifiedModelComparator
from master_arbeit_Di.explore.GMpreprocess import GMpreprocess

In [ ]:
# =============================================================================
# CONFIGURATION SECTION - EDIT THESE TO CUSTOMIZE YOUR ANALYSIS
# =============================================================================
# Dataset and directories
DATASET_PATH = r"..\\..\\explore_data\\G6M2.parquet"
PREPROCESS_OUTPUT_DIR = r"..\\..\\explore_data\\output"
PLOTS_OUTPUT_DIR = r"..\\..\\plots\\huber\\G6M2_comparison"
SAVE_PLOTS = True

# Reference condition configurations: Low, Medium, High
REF_CONFIGS = {
    "Low": {
        "Iref": 0.28,
        "Tref": 57,
        "OHref": 10,
        "gt_file": r"..\\..\\ground_truth\\output_backup\\gt_raw_processed\\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10__daily_regression_full_coverage.csv",
    },
    "Medium": {
        "Iref": 1.0,
        "Tref": 58,
        "OHref": 33,
        "gt_file": r"..\\..\\ground_truth\\output_backup\\gt_raw_processed\\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33__daily_regression_full_coverage.csv",
    },
    "High": {
        "Iref": 1.31,
        "Tref": 58,
        "OHref": 18,
        "gt_file": r"..\\..\\ground_truth\\output_backup\\gt_raw_processed\\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18__daily_regression_full_coverage.csv",
    },
}

# Model fitting common config (shared across all models)
COMMON_CONFIG = {
    "Iref": [cfg["Iref"] for cfg in REF_CONFIGS.values()],
    "Tref": 60,
    "OHref": 72,
    "ref_config": REF_CONFIGS,
    "len_interval": 2,
    "slide": 1,
    "min_num_data_required_for_fit": 300,
    "threshold": 1e6,
    "i_off": 0.1,
    "u_off": 1.3,
    "plot_fit": 0,
    "data_filter_i_min": 0.1,
    "data_filter_U_min": 1.4,
    "data_filter_U_max": 2.3,
    "data_filter_T_min": 50,
    "data_filter_T_max": 65,
    "data_filter_h_since_last_start_min": 0.5,
}

# Comparison settings
SHOW_GT_METRICS = True
SHOW_ALL_COND_METRICS = True
REF_ORDER = list(REF_CONFIGS.keys())

In [ ]:
# =============================================================================
# 1. DATA LOADING & PREPROCESSING
# =============================================================================
print("=" * 80)
print("STEP 1: Data Loading & Preprocessing")
print("=" * 80)
preprocessor = GMpreprocess(file_path=DATASET_PATH, output_dir=PREPROCESS_OUTPUT_DIR)
data = preprocessor.run()
dataset_name = preprocessor.name
print(f"Dataset: {dataset_name}, shape: {data.shape}")
print(f"Time range: {data.index.min()} -> {data.index.max()}\n")

shared_pre = Urc1.preprocess_once(
    data,
    i_off=COMMON_CONFIG["i_off"],
    u_off=COMMON_CONFIG["u_off"],
    data_filter_i_min=COMMON_CONFIG["data_filter_i_min"],
    data_filter_U_min=COMMON_CONFIG["data_filter_U_min"],
    data_filter_U_max=COMMON_CONFIG["data_filter_U_max"],
    data_filter_T_min=COMMON_CONFIG["data_filter_T_min"],
    data_filter_T_max=COMMON_CONFIG["data_filter_T_max"],
)
print(f"Shared preprocessed rows: {len(shared_pre)}\n")

STEP 1: Data Loading & Preprocessing
=== 1. Loading & Preprocessing: G6M2 ===
>> Data loaded successfully.
   [Detected] Temperature Col: 'Temp_Module_12' -> ID: '2'
>> Columns renamed to standard format.
>> Columns filtered. Retained: ['currentDensity', 'temperature', 'voltage']

=== 3. Saving Preprocessd data in .parquet format ===
>> ✅ Final Results saved successfully to:
   ..\\explore_data\\output\G6M2_20260430_112347.parquet

=== GMpreprocess Pipeline Completed Successfully ===
Dataset: G6M2, shape: (1096020, 3)
Time range: 2023-07-06 00:00:00 -> 2025-08-05 09:29:00

[preprocess_once] 1096020 -> 359353 points.
Shared preprocessed rows: 359353



In [ ]:
# =============================================================================
# 2. TRAIN MODELS
# =============================================================================
print("=" * 80)
print("STEP 2: Model Training")
print("=" * 80)

models = {}
print("\n  -> Training Baseline (OLS) model...")
urc_baseline = Urc1(
    data=data,
    name=dataset_name,
    preprocessed_data=shared_pre,
    nonlinear_term="I2",
    **COMMON_CONFIG,
)
models["Baseline (I2)"] = urc_baseline
print("  ✓ Baseline training completed")

print("\n  -> Training Huber model...")
urc_huber = Urc1BaseHuber(
    data=data,
    name=dataset_name,
    epsilon=1.00,
    alpha=0.0,
    preprocessed_data=shared_pre,
    **COMMON_CONFIG,
)
models["Huber"] = urc_huber
print("  ✓ Huber training completed")

print("\n  -> Training Ridge model...")
urc_ridge = Urc1BaseHuber(
    data=data,
    name=dataset_name,
    epsilon=1e6,
    alpha=0.1,
    preprocessed_data=shared_pre,
    **COMMON_CONFIG,
)
models["Ridge"] = urc_ridge
print("  ✓ Ridge training completed")

print("\n  -> Training Huber+Ridge model...")
urc_hr = Urc1BaseHuber(
    data=data,
    name=dataset_name,
    epsilon=1.35,
    alpha=0.1,
    preprocessed_data=shared_pre,
    **COMMON_CONFIG,
)
models["Huber+Ridge"] = urc_hr
print("  ✓ Huber+Ridge training completed")

print(f"\n✓ {len(models)} models trained successfully\n")

STEP 2: Model Training

  -> Training Baseline (OLS) model...
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Fitting Stats: 218 intervals low data, 0 fit failed.
468 out of 762 fitting results are reliable.
  ✓ Baseline training completed

  -> Training Huber model...
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
[Huber eps=1.0 alpha=0.0] Fitting Stats: 218 intervals low data, 0 fit failed.
464 out of 762 fitting results are reliable.
  [Huber] epsilon=1.0, alpha=0.0
  ✓ Huber training completed

  -> Training Ridge model...
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
[Ridge eps=1000000.0 alpha=0.1] Fitting Stats: 218 intervals low data, 0 fit failed.
537 out of 762 fitting results are reliable.
  [Ridge] epsilon=1000000.0, alpha=0.1
  ✓ Ridge training completed

  -> Training Huber+Ridge model...
Using shared preprocessed data (359353 poin

In [ ]:
# =============================================================================
# 3. COMPARATOR SETUP & GT LOADING
# =============================================================================
print("\n" + "=" * 80)
print("STEP 3: Initialize Comparator & Load Ground Truth")
print("=" * 80)

comparator = UnifiedModelComparator(models)
print(f"✓ UnifiedModelComparator initialized with {len(models)} models\n")

gt_loaded_count = 0
for ref_name, ref_cfg in REF_CONFIGS.items():
    iref = ref_cfg["Iref"]
    gt_file = ref_cfg["gt_file"]
    gt_path = Path(gt_file)
    if not gt_path.is_absolute():
        gt_path = Path.cwd() / gt_path
    gt_path = gt_path.resolve()
    print(f"Looking for GT file: {gt_path}")

    if gt_path.exists():
        try:
            gt_data = pd.read_csv(gt_path, index_col=0, parse_dates=True)
            colmap = {str(c).strip().lower(): c for c in gt_data.columns}
            candidate_cols = ["gt_uref_regression", "voltage", "uref", "gt_uref"]
            selected_col = None
            for c in candidate_cols:
                if c in colmap:
                    selected_col = colmap[c]
                    break

            if selected_col is not None:
                gt_series = gt_data[selected_col]
            elif len(gt_data.columns) == 1:
                gt_series = gt_data.iloc[:, 0]
                selected_col = gt_data.columns[0]
            else:
                raise ValueError(
                    f"Cannot identify voltage column in {gt_path}. Columns: {gt_data.columns.tolist()}"
                )

            gt_series = gt_series.dropna()
            comparator.set_ground_truth(gt_series, iref=iref)
            gt_loaded_count += 1
            print(
                f"  ✓ Loaded GT for {ref_name} (Iref={iref}): {len(gt_series)} points [column: {selected_col}]"
            )
        except Exception as e:
            print(f"  ✗ Failed to load GT for {ref_name}: {e}")
    else:
        print(f"  ⚠ GT file NOT found: {gt_path}")

has_gt = gt_loaded_count > 0 and SHOW_GT_METRICS
print(f"\n{'=' * 80}")
print(f"GT status: {gt_loaded_count}/{len(REF_CONFIGS)} reference conditions loaded")
print(f"GT metrics will be {'ENABLED ✓' if has_gt else 'DISABLED'}")
print(f"{'=' * 80}\n")


STEP 3: Initialize Comparator & Load Ground Truth
✓ UnifiedModelComparator initialized with 4 models

Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10__daily_regression_full_coverage.csv
  ✓ Loaded GT for Low (Iref=0.28): 758 points [column: gt_uref_regression]
Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33__daily_regression_full_coverage.csv
  ✓ Loaded GT for Medium (Iref=1.0): 758 points [column: gt_uref_regression]
Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18__daily_regression_full_coverage.csv
  ✓ Loaded GT for High (Iref=1.31): 758 points [column: gt_uref_regression]

GT status: 3/3 referen

In [ ]:
# =============================================================================
# 4. REFERENCE-SPECIFIC ANALYSIS
# =============================================================================
print("=" * 80)
print("STEP 4: Reference-Specific Metrics & Comparison")
print("=" * 80)

rate_tables = []
for ref_name in REF_ORDER:
    ref_cfg = REF_CONFIGS[ref_name]
    iref = ref_cfg["Iref"]
    tref = ref_cfg["Tref"]
    ohref = ref_cfg["OHref"]

    print(f"\n▓▓▓ REFERENCE CONDITION: {ref_name} (Iref={iref}, Tref={tref}, OHref={ohref}) ▓▓▓")

    df_metrics = comparator.compare_all(
        i_target=iref,
        outlier_threshold_method="2rmse",
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
        include_gt_metrics=has_gt,
    )
    display(Markdown(df_metrics.to_markdown(index=False)))

    rate_tables.append(
        df_metrics[["Model Name", "Target Current (A/cm2)", "Degradation Rate (uV/h)", "Slope Sigma (uV/h)"]].assign(
            Reference=ref_name
        )
    )

    try:
        comparator.plot_interactive_trends(
            target_i=iref,
            show_gt=has_gt,
            save=SAVE_PLOTS,
            output_dir=PLOTS_OUTPUT_DIR,
            uncertainty_style="band",
            uncertainty_opacity=0.12,
            show_series_line=True,
            rate_precision=6,
        )
        print("✓ Trend plot finished")
    except Exception as e:
        print(f"⚠ Trend plot failed: {e}")

    comparator.print_comparison_report(
        i_target=iref,
        include_gt_metrics=has_gt,
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
    )

STEP 4: Reference-Specific Metrics & Comparison

▓▓▓ REFERENCE CONDITION: Low (Iref=0.28, Tref=57, OHref=10) ▓▓▓


| Model Name    |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:--------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline (I2) |                     0.28 |              6.203 |               468 |                   2.15303 |       3.532 |                0     |               0.966 |              21 |          4.49 |              23.214 |          1.814 |               5.071 |           13.217 |                       539 | all_fitted   |          8.004 |         7.205 |                  468 |
| Huber         |                     0.28 |              8.432 |               464 |                   2.1066  |       3.413 |                0.039 |               0.972 |              25 |          5.39 |              18.869 |          1.25  |               5.307 |           13.252 |                       539 | all_fitted   |          8.032 |         7.234 |                  464 |
| Ridge         |                     0.28 |             10.49  |               537 |                   2.11636 |       6.447 |                0.056 |               0.951 |              10 |          1.86 |             108.857 |          1.681 |               4.105 |            4.57  |                       539 | all_fitted   |          8.585 |         6.02  |                  537 |
| Huber+Ridge   |                     0.28 |             10.793 |               538 |                   2.03218 |       6.427 |                0.049 |               0.956 |               3 |          0.56 |             106.366 |          1.259 |               4.038 |            4.491 |                       539 | all_fitted   |          9.119 |         6.986 |                  538 |

✓ Trend plot finished
  MODEL COMPARISON REPORT @ 0.28 A/cm²

📊 Baseline (I2)
------------------------------------------------------------
  Data Points:      468
  Deg Rate:         2.153026 μV/h
  RMSE:             3.532 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.966
  Outliers:         21 (4.5%)
  Max Residual:     23.214 mV
  Mean SE:          1.814 mV
  Cond (Median):    10^5.1
  Cond Scope:       all_fitted (n=539)
  GT RMSE:          8.004 mV (n=468)
  GT MAE:           7.205 mV

📊 Huber
------------------------------------------------------------
  Data Points:      464
  Deg Rate:         2.106604 μV/h
  RMSE:             3.413 mV
  Slope Sigma:      0.039 μV/h
  Mono (Rank):      0.972
  Outliers:         25 (5.4%)
  Max Residual:     18.869 mV
  Mean SE:          1.250 mV
  Cond (Median):    10^5.3
  Cond Scope:       all_fitted (n=539)
  GT RMSE:          8.032 mV (n=464)
  GT MAE:           7.234 mV

📊 Ridge
----------------------------------------------------

| Model Name    |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:--------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline (I2) |                        1 |              6.203 |               468 |                   5.84556 |       6.83  |                0     |               0.983 |              21 |          4.49 |              21.578 |          1.851 |               5.071 |           13.217 |                       539 | all_fitted   |         14.503 |        12.942 |                  468 |
| Huber         |                        1 |              8.432 |               464 |                   5.96486 |       6.967 |                0.067 |               0.982 |              22 |          4.74 |              25.14  |          1.297 |               5.307 |           13.252 |                       539 | all_fitted   |         14.911 |        13.38  |                  464 |
| Ridge         |                        1 |             10.49  |               537 |                   5.91753 |      51.452 |                0.271 |               0.808 |              46 |          8.57 |             218.506 |          1.864 |               4.105 |            4.57  |                       539 | all_fitted   |         50.79  |        24.522 |                  537 |
| Huber+Ridge   |                        1 |             10.793 |               538 |                   6.18014 |      67.175 |                0.359 |               0.844 |              42 |          7.81 |             362.792 |          1.41  |               4.038 |            4.491 |                       539 | all_fitted   |         66.431 |        28.835 |                  538 |

✓ Trend plot finished
  MODEL COMPARISON REPORT @ 1.0 A/cm²

📊 Baseline (I2)
------------------------------------------------------------
  Data Points:      468
  Deg Rate:         5.845556 μV/h
  RMSE:             6.830 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.983
  Outliers:         21 (4.5%)
  Max Residual:     21.578 mV
  Mean SE:          1.851 mV
  Cond (Median):    10^5.1
  Cond Scope:       all_fitted (n=539)
  GT RMSE:          14.503 mV (n=468)
  GT MAE:           12.942 mV

📊 Huber
------------------------------------------------------------
  Data Points:      464
  Deg Rate:         5.964859 μV/h
  RMSE:             6.967 mV
  Slope Sigma:      0.067 μV/h
  Mono (Rank):      0.982
  Outliers:         22 (4.7%)
  Max Residual:     25.140 mV
  Mean SE:          1.297 mV
  Cond (Median):    10^5.3
  Cond Scope:       all_fitted (n=539)
  GT RMSE:          14.911 mV (n=464)
  GT MAE:           13.380 mV

📊 Ridge
-------------------------------------------------

| Model Name    |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:--------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline (I2) |                     1.31 |              6.203 |               468 |                   8.46857 |      10.333 |                0     |               0.982 |              16 |          3.42 |              44.804 |          2.249 |               5.071 |           13.217 |                       539 | all_fitted   |         13.937 |        10.982 |                  468 |
| Huber         |                     1.31 |              8.432 |               464 |                   8.66018 |      10.782 |                0.097 |               0.979 |              18 |          3.88 |              57.114 |          1.861 |               5.307 |           13.252 |                       539 | all_fitted   |         14.57  |        11.502 |                  464 |
| Ridge         |                     1.31 |             10.49  |               537 |                   8.12764 |      76.986 |                0.306 |               0.808 |              46 |          8.57 |             326.333 |          2.089 |               4.105 |            4.57  |                       539 | all_fitted   |         75.016 |        36.04  |                  537 |
| Huber+Ridge   |                     1.31 |             10.793 |               538 |                   8.50248 |      97.341 |                0.394 |               0.842 |              42 |          7.81 |             530.831 |          1.606 |               4.038 |            4.491 |                       539 | all_fitted   |         96.829 |        35.567 |                  538 |

✓ Trend plot finished
  MODEL COMPARISON REPORT @ 1.31 A/cm²

📊 Baseline (I2)
------------------------------------------------------------
  Data Points:      468
  Deg Rate:         8.468567 μV/h
  RMSE:             10.333 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.982
  Outliers:         16 (3.4%)
  Max Residual:     44.804 mV
  Mean SE:          2.249 mV
  Cond (Median):    10^5.1
  Cond Scope:       all_fitted (n=539)
  GT RMSE:          13.937 mV (n=468)
  GT MAE:           10.982 mV

📊 Huber
------------------------------------------------------------
  Data Points:      464
  Deg Rate:         8.660182 μV/h
  RMSE:             10.782 mV
  Slope Sigma:      0.097 μV/h
  Mono (Rank):      0.979
  Outliers:         18 (3.9%)
  Max Residual:     57.114 mV
  Mean SE:          1.861 mV
  Cond (Median):    10^5.3
  Cond Scope:       all_fitted (n=539)
  GT RMSE:          14.570 mV (n=464)
  GT MAE:           11.502 mV

📊 Ridge
----------------------------------------------

In [ ]:
# =============================================================================
# 5. CROSS-MODEL DIAGNOSTICS
# =============================================================================
print("\n" + "=" * 80)
print("STEP 5: Cross-Model Diagnostics")
print("=" * 80)

try:
    print("\n-> Plotting fit quality (RMSE & R² distributions)...")
    comparator.plot_fit_quality(save=SAVE_PLOTS)
    print("✓ Fit quality completed")
except Exception as e:
    print(f"⚠ Fit quality plot failed: {e}")

try:
    print("\n-> Plotting coefficient diagnostics...")
    comparator.plot_coefficient_diagnostic(save=SAVE_PLOTS, include_c6=False)
    print("✓ Coefficient diagnostic completed")
except Exception as e:
    print(f"⚠ Coefficient diagnostic failed: {e}")

try:
    print("\n-> Plotting coverage Gantt...")
    comparator.plot_coverage_gantt(save=SAVE_PLOTS)
    print("✓ Coverage Gantt completed")
except Exception as e:
    print(f"⚠ Coverage Gantt failed: {e}")


STEP 5: Cross-Model Diagnostics

-> Plotting fit quality (RMSE & R² distributions)...
✓ Fit quality completed

-> Plotting coefficient diagnostics...
✓ Coefficient diagnostic completed

-> Plotting coverage Gantt...
✓ Coverage Gantt completed


In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)
print(f"\n✓ Trained models: {len(models)}")
print(f"✓ Reference conditions analyzed: {len(REF_CONFIGS)}")
print(f"✓ Ground truth data: {'Loaded ✓' if has_gt else 'Not available'}")
print(f"✓ Output plots: {PLOTS_OUTPUT_DIR}/" if SAVE_PLOTS else "✓ Plots displayed (not saved)")
print("\nKey settings:")
print(f"  - All condition metrics: {SHOW_ALL_COND_METRICS}")
print(f"  - GT metrics: {SHOW_GT_METRICS}")
print(f"  - Save plots: {SAVE_PLOTS}")


ANALYSIS COMPLETE

✓ Trained models: 4
✓ Reference conditions analyzed: 3
✓ Ground truth data: Loaded ✓
✓ Output plots: plots\\huber\\G6M2_comparison/

Key settings:
  - All condition metrics: True
  - GT metrics: True
  - Save plots: True
